# 🛰️ Satellite Vegetation Monitor
### Hands-On Activity · Lesson 6 — Geospatial AI and Satellite Data
**AI for Sustainable Development · ESDU @ AUB · karianet.org**

---

In this activity you will use **Google Earth Engine (GEE)** via its Python API to:

1. Authenticate and connect to GEE's satellite data archive
2. Load **Sentinel-2** imagery for an agricultural region in Lebanon
3. Calculate **NDVI** (vegetation health) and **NDWI** (water content) indices
4. Visualise how these indices change **across seasons**
5. Interpret what those changes tell us about crop stress

**Everything runs in Google Colab** — no software to install.  
You will need a **free Google Earth Engine account** (instructions in Part 1).  
Run cells **top to bottom** with **Shift + Enter**.


## Step 0 — Enter your details

In [ ]:
your_name  = "Your Full Name"   # ← replace
your_email = "your@email.com"   # ← replace
print(f"Name : {your_name}")
print(f"Email: {your_email}")


## Part 1 — Set Up Google Earth Engine

### One-time account setup (5 minutes)
If you don't already have a GEE account:
1. Go to **https://earthengine.google.com/**
2. Click **"Get Started"** → sign in with a Google account
3. Click **"Register a Noncommercial or Commercial Cloud Project"**
4. Choose **"Unpaid usage"** → **"Academia & Research"**
5. Follow the prompts — approval is usually instant for academic users

### Then run the cell below to install and authenticate


In [ ]:
# Install the Earth Engine Python API
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "earthengine-api",
                "geemap", "matplotlib", "numpy", "--quiet"], capture_output=True)

import ee, geemap, json, hashlib, re, os, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from datetime import datetime
warnings.filterwarnings("ignore")

print("✅ Libraries installed. Now authenticate with Earth Engine below.")


In [ ]:
# Authenticate — this will open a Google sign-in window
# Sign in with the same account you registered on earthengine.google.com
try:
    ee.Initialize()
    print("✅ Already authenticated. Earth Engine ready.")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("✅ Authenticated and initialised.")


## Part 2 — Concepts: What Are NDVI and NDWI?

Before we calculate anything, let's understand what these indices measure.

### NDVI — Normalised Difference Vegetation Index
Measures how "green" and healthy vegetation is using light reflectance:

```
NDVI = (Near-Infrared − Red) / (Near-Infrared + Red)
```

| NDVI value | Meaning |
|------------|---------|
| < 0        | Water, bare rock, cloud |
| 0.0 – 0.2  | Bare soil, sparse cover |
| 0.2 – 0.5  | Moderate vegetation, stressed crops |
| 0.5 – 1.0  | Dense, healthy vegetation |

### NDWI — Normalised Difference Water Index
Measures water content in vegetation:

```
NDWI = (Green − Near-Infrared) / (Green + Near-Infrared)
```

High NDWI values indicate more water in the plant tissue — important for
detecting drought stress before it becomes visible to the naked eye.

### Why does this matter for agriculture?
A field whose NDVI suddenly drops from 0.6 to 0.3 between two satellite
passes may be experiencing drought, pest damage, or disease — weeks before
a farmer walking the field would notice anything.


## Part 3 — Define Your Study Area

The default area is the **Bekaa Valley**, Lebanon's main agricultural region.
You can change the coordinates to any location you're interested in.

Run the cell to display the area on an interactive map.


In [ ]:
# ── STUDY AREA ──────────────────────────────────────────────────────────
# Default: Bekaa Valley, Lebanon
# Format: [min_lon, min_lat, max_lon, max_lat]
STUDY_AREA = [35.85, 33.55, 36.30, 33.90]

# ── STUDY PERIOD ────────────────────────────────────────────────────────
YEAR = 2023  # change to any year GEE has data for (2017–present for S2)

# Build GEE geometry
roi = ee.Geometry.Rectangle(STUDY_AREA)

# Display on interactive map
Map = geemap.Map(center=[(STUDY_AREA[1]+STUDY_AREA[3])/2,
                          (STUDY_AREA[0]+STUDY_AREA[2])/2],
                  zoom=9)
Map.addLayer(roi, {"color": "red"}, "Study Area")
Map.addLayerControl()
display(Map)
print(f"\n✅ Study area defined: {STUDY_AREA}")
print(f"   Year: {YEAR}")


## Part 4 — Load Sentinel-2 Imagery and Calculate Indices

Sentinel-2 is a freely available satellite constellation from the European Space Agency.
It revisits most locations every **5 days** at **10-metre resolution**.

We'll load cloud-free composites for **four seasons** and calculate NDVI and NDWI for each.


In [ ]:
def get_seasonal_composite(roi, year, month_start, month_end, season_name):
    """
    Returns cloud-masked Sentinel-2 median composite with NDVI and NDWI
    for a given season window.
    """
    start = f"{year}-{month_start:02d}-01"
    end   = f"{year}-{month_end:02d}-28"

    collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                  .filterDate(start, end)
                  .filterBounds(roi)
                  .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
                  .map(lambda img: img.divide(10000)))  # scale reflectance

    composite = collection.median().clip(roi)

    # Band naming: B4=Red, B8=NIR, B3=Green
    ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = composite.normalizedDifference(["B3", "B8"]).rename("NDWI")

    result = composite.addBands([ndvi, ndwi])
    count  = collection.size().getInfo()
    print(f"  {season_name:<12}: {count} images used  ({start} → {end})")
    return result, ndvi, ndwi, season_name

seasons = [
    (YEAR, 1,  3,  "Winter"),
    (YEAR, 4,  6,  "Spring"),
    (YEAR, 7,  9,  "Summer"),
    (YEAR, 10, 12, "Autumn"),
]

print(f"Loading Sentinel-2 composites for {YEAR}...")
composites = []
for (yr, m1, m2, name) in seasons:
    comp, ndvi, ndwi, sname = get_seasonal_composite(roi, yr, m1, m2, name)
    composites.append({"name": sname, "composite": comp,
                        "ndvi": ndvi, "ndwi": ndwi})

print("\n✅ All seasonal composites ready.")


## Part 5 — Exercise A: Visualise NDVI Across Seasons

Run the cell below to display NDVI maps for all four seasons side by side.
Compare them — which season shows the highest vegetation cover?
Which shows drought stress?


In [ ]:
# ── EXERCISE A — Seasonal NDVI maps ──────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle(f"NDVI — {STUDY_AREA} — {YEAR}", fontsize=14, fontweight="bold")

ndvi_cmap = plt.cm.RdYlGn  # red (low) → yellow → green (high)
stored_ndvi = {}

for ax, season_data in zip(axes, composites):
    sname = season_data["name"]
    try:
        # Sample NDVI values on a grid over the ROI
        ndvi_img = season_data["ndvi"]
        sample = ndvi_img.sample(region=roi, scale=1000, numPixels=500,
                                 seed=42, geometries=False).getInfo()
        vals = [f["properties"]["NDVI"] for f in sample["features"]
                if f["properties"].get("NDVI") is not None]
        mean_ndvi = float(np.mean(vals)) if vals else 0.0
        stored_ndvi[sname] = {"mean": round(mean_ndvi, 3), "values": vals[:100]}

        # Colour-mapped scatter to represent spatial variation
        ax.scatter([i % 20 for i in range(len(vals))],
                   [i // 20 for i in range(len(vals))],
                   c=vals, cmap=ndvi_cmap, vmin=-0.2, vmax=0.8,
                   s=60, marker="s", linewidths=0)
        ax.set_title(f"{sname}\nMean NDVI: {mean_ndvi:.3f}", fontsize=11)
        ax.axis("off")
    except Exception as ex:
        ax.set_title(f"{sname}\n(data unavailable)", fontsize=10)
        ax.text(0.5, 0.5, str(ex)[:60], transform=ax.transAxes,
                ha="center", va="center", fontsize=7, color="red", wrap=True)
        ax.axis("off")
        stored_ndvi[sname] = {"mean": 0.0, "values": []}

sm = plt.cm.ScalarMappable(cmap=ndvi_cmap, norm=plt.Normalize(vmin=-0.2, vmax=0.8))
sm.set_array([])
fig.colorbar(sm, ax=axes, shrink=0.6, label="NDVI")
plt.tight_layout()
plt.savefig("ndvi_seasonal.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ NDVI seasonal maps saved as ndvi_seasonal.png")


## Part 6 — Exercise B: NDVI Time-Series Chart

A map shows you where things are — a time series shows you **when** things change.
Run the cell below to plot mean NDVI by season as a line chart.


In [ ]:
# ── EXERCISE B — NDVI time-series ────────────────────────────────────────
season_names = [s["name"] for s in composites]
mean_ndvis   = [stored_ndvi.get(s, {}).get("mean", 0) for s in season_names]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(season_names, mean_ndvis, "o-", color="#2ecc71", linewidth=2.5,
        markersize=9, markerfacecolor="#27ae60")
for x, y in zip(season_names, mean_ndvis):
    ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=10)

ax.axhline(0.2, color="orange", linestyle="--", alpha=0.6, label="Vegetation threshold (0.2)")
ax.axhline(0.5, color="green",  linestyle="--", alpha=0.4, label="Dense vegetation (0.5)")
ax.set_title(f"Mean NDVI by Season — Bekaa Valley {YEAR}", fontsize=13, fontweight="bold")
ax.set_ylabel("Mean NDVI")
ax.set_ylim(-0.1, 0.9)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("ndvi_timeseries.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Time-series chart saved as ndvi_timeseries.png")

# Print summary table
print("\nSeasonal NDVI summary:")
print(f"{'Season':<12} {'Mean NDVI':<12} {'Interpretation'}")
print("-" * 50)
for name, val in zip(season_names, mean_ndvis):
    interp = ("Dense vegetation" if val > 0.5 else
              "Moderate vegetation" if val > 0.3 else
              "Sparse/stressed" if val > 0.1 else
              "Bare soil / water")
    print(f"{name:<12} {val:<12.3f} {interp}")


## Part 7 — Reflection Questions


In [ ]:
# Reflection 1
# Looking at your NDVI time-series chart, describe what you observe.
# Which season shows the most vegetation stress, and what might cause this
# in the Bekaa Valley or in a similar MENA agricultural context?

reflection_1 = """
Write your answer here. Reference specific values from your chart.
"""
print(reflection_1.strip() if len(reflection_1.strip()) > 20 else "❌ Write at least 2 sentences referencing your chart.")


In [ ]:
# Reflection 2
# How could a farmer or an agricultural NGO use seasonal NDVI monitoring
# to make better decisions? Give one concrete example with a specific action
# triggered by a specific NDVI reading.

reflection_2 = """
Write your answer here.
"""
print(reflection_2.strip() if len(reflection_2.strip()) > 20 else "❌ Write at least 2 sentences.")


## Part 8 — Submit Your Activity

In [ ]:
import ipywidgets as widgets, base64
from IPython.display import display

_btn = widgets.Button(description="Submit Activity", button_style="success",
                      layout=widgets.Layout(width="220px", height="40px"))
_out = widgets.Output()

def _submit(b):
    _out.clear_output()
    with _out:
        errs = []
        if len(your_name.strip()) < 3 or your_name.strip() == "Your Full Name":
            errs.append("❌ Enter your full name in Step 0.")
        if "@" not in your_email or your_email.strip() == "your@email.com":
            errs.append("❌ Enter a valid email in Step 0.")
        if not stored_ndvi or all(v.get("mean", 0) == 0 for v in stored_ndvi.values()):
            errs.append("❌ Complete Exercise A (NDVI maps must be generated).")
        if not os.path.exists("ndvi_timeseries.png"):
            errs.append("❌ Complete Exercise B (time-series chart must be saved).")
        if len(reflection_1.strip()) < 30:
            errs.append("❌ Reflection 1 is too short.")
        if len(reflection_2.strip()) < 30:
            errs.append("❌ Reflection 2 is too short.")

        if errs:
            for e in errs: print(e)
            print("\nFix the items above and click Submit Activity again.")
            return

        charts = {}
        for fname in ["ndvi_seasonal.png", "ndvi_timeseries.png"]:
            if os.path.exists(fname):
                with open(fname, "rb") as f:
                    charts[fname] = base64.b64encode(f.read()).decode()

        payload = {
            "activity": "Satellite Vegetation Monitor",
            "lesson": "Lesson 6 — Geospatial AI and Satellite Data",
            "name": your_name.strip(),
            "email": your_email.strip(),
            "study_area": STUDY_AREA,
            "year": YEAR,
            "ndvi_seasonal": stored_ndvi,
            "charts_base64": charts,
            "reflection_1": reflection_1.strip(),
            "reflection_2": reflection_2.strip(),
            "timestamp": datetime.utcnow().isoformat() + "Z",
        }
        raw = json.dumps({k: v for k, v in payload.items() if k != "charts_base64"},
                         sort_keys=True, ensure_ascii=False, default=str)
        payload["checksum"] = hashlib.sha256(raw.encode()).hexdigest()

        slug  = re.sub(r"[^a-zA-Z0-9]+", "_", your_name.strip().lower()).strip("_") or "student"
        fname = f"submission_{slug}.json"
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False, default=str)

        print(f"🎉 Submission ready: {fname}")
        print(f"   Seasons analysed: {list(stored_ndvi.keys())}")
        print(f"   Checksum: {payload['checksum'][:16]}...")
        try:
            from google.colab import files
            files.download(fname)
            print("\n⬇️  Download started — check your Downloads folder.")
            print("📌 Upload this file to the assignment box on Karianet.")
        except ImportError:
            print("\nℹ️  Not in Colab — find file in the Files panel.")

_btn.on_click(_submit)
display(_btn, _out)


---
## Submission Checklist
- [ ] Name and email entered
- [ ] GEE authentication successful (Part 1)
- [ ] Seasonal composites loaded (Part 4)
- [ ] Exercise A: NDVI seasonal maps generated
- [ ] Exercise B: Time-series chart generated
- [ ] Both reflections written referencing your actual data
- [ ] Submit button clicked and `submission_[yourname].json` downloaded
- [ ] File uploaded to the assignment box on Karianet

---
*Satellite Vegetation Monitor · AI for Sustainable Development · Lesson 6 · ESDU @ AUB*
